# Projet Challenge 2 – Variables v2 + Module 3 : modélisation

**Partie A – Variables v2.** Corrections issues de l'analyse des Modules 1 et 2 (voir le journal des imprévus dans Notion) :
- suppression des 3 doublons exacts ;
- `ville_differente_prec` (signal inversé) remplacée par `nouvelle_ville` ;
- ajout de vélocités contextuelles : `nb_etranger_24h`, `nb_nouveaux_marchands_24h`, `montant_24h`.

**Partie B – Module 3.** Stratégie contre le déséquilibre (comparée en validation), modèles supervisés
(régression logistique pondérée, XGBoost) et non supervisés (Isolation Forest, autoencodeur, LOF en option),
puis tableau comparatif sur la **validation** (2018). Le test (2019) n'est pas touché.

## 0. Chargement

In [ ]:
!pip -q install xgboost imbalanced-learn

from google.colab import drive
drive.mount("/content/drive")

import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DOSSIER = "/content/drive/MyDrive/fraude_tabformer"
ech = pd.read_parquet(f"{DOSSIER}/echantillon_500_users.parquet")
print(f"{len(ech):,} transactions chargées")

---
# Partie A – Variables v2

Toute la construction est regroupée dans une fonction : elle servira telle quelle plus tard pour l'API.

In [ ]:
def construire_variables(ech):
    """Variables sans fuite : chaque transaction n'utilise que le passé de sa carte."""
    data = ech.drop_duplicates().copy()
    data["carte"] = data["User"].astype(str) + "_" + data["Card"].astype(str)
    data = data.sort_values(["carte", "date"], kind="mergesort").reset_index(drop=True)

    # --- temporelles
    h = data["date"].dt.hour + data["date"].dt.minute / 60
    data["heure_sin"] = np.sin(2 * np.pi * h / 24)
    data["heure_cos"] = np.cos(2 * np.pi * h / 24)
    data["jour_semaine"] = data["date"].dt.dayofweek.astype("int8")
    data["week_end"] = (data["jour_semaine"] >= 5).astype("int8")
    data["nuit"] = (h < 6).astype("int8")

    # --- transaction
    data["montant"] = data["Amount"].astype("float32")
    data["remboursement"] = (data["montant"] < 0).astype("int8")
    data["log_montant"] = np.sign(data["montant"]) * np.log1p(data["montant"].abs())
    canal = data["Use Chip"].astype(str)
    data["en_ligne"] = canal.str.contains("Online").astype("int8")
    data["bande"] = canal.str.contains("Swipe").astype("int8")
    data["etranger"] = ((data["Merchant State"].astype(str).str.len() > 2) & (data["en_ligne"] == 0)).astype("int8")
    err = data["Errors?"].astype(str)
    for nom, motif in {"err_cvv": "Bad CVV", "err_expiration": "Bad Expiration", "err_numero": "Bad Card Number",
                       "err_pin": "Bad PIN", "err_solde": "Insufficient Balance", "err_technique": "Technical Glitch",
                       "err_code_postal": "Bad Zipcode"}.items():
        data[nom] = err.str.contains(motif, regex=False).astype("int8")
    data["une_erreur"] = data[[c for c in data.columns if c.startswith("err_")]].max(axis=1).astype("int8")
    mcc = data["MCC"].astype(int)
    data["mcc_groupe"] = np.select([mcc.between(3000, 3299), mcc.between(3351, 3441), mcc.between(3501, 3999)],
                                   ["aerien", "location_voiture", "hotel"], default=mcc.astype(str))

    # --- nouveauté (premier achat de la carte chez ce marchand / catégorie / ville)
    data["nouveau_marchand"] = (data.groupby(["carte", "Merchant Name"]).cumcount() == 0).astype("int8")
    data["nouvelle_categorie"] = (data.groupby(["carte", "mcc_groupe"]).cumcount() == 0).astype("int8")
    data["nouvelle_ville"] = ((data.groupby(["carte", "Merchant City"], observed=True).cumcount() == 0)
                              & (data["en_ligne"] == 0)).astype("int8")          # v2 : remplace ville_differente_prec

    # --- comportementales : fenêtres glissantes sur le passé (closed="left" exclut la transaction en cours)
    g = data.groupby("carte", sort=False)
    data["h_depuis_prec"] = (g["date"].diff().dt.total_seconds() / 3600).astype("float32")
    data["part_en_ligne_passee"] = (g["en_ligne"].cumsum() - data["en_ligne"]) / g.cumcount().replace(0, np.nan)
    data["montant_pos"] = data["montant"].clip(lower=0)

    roul = data.set_index("date").groupby("carte", sort=False)
    def fenetre(col, taille, fonction):
        return getattr(roul[col].rolling(taille, closed="left"), fonction)().to_numpy()

    data["nb_tx_1h"] = np.nan_to_num(fenetre("montant", "1h", "count")).astype("int16")
    data["nb_tx_24h"] = np.nan_to_num(fenetre("montant", "24h", "count")).astype("int16")
    data["nb_tx_30j"] = np.nan_to_num(fenetre("montant", "30D", "count")).astype("int16")
    data["nb_err_24h"] = np.nan_to_num(fenetre("une_erreur", "24h", "sum")).astype("int16")
    data["nb_etranger_24h"] = np.nan_to_num(fenetre("etranger", "24h", "sum")).astype("int16")              # v2
    data["nb_nouveaux_marchands_24h"] = np.nan_to_num(fenetre("nouveau_marchand", "24h", "sum")).astype("int16")  # v2
    data["montant_24h"] = np.nan_to_num(fenetre("montant_pos", "24h", "sum")).astype("float32")             # v2
    data["moy_30j"] = fenetre("montant", "30D", "mean").astype("float32")
    data["ratio_montant_30j"] = (data["montant"] / (data["moy_30j"].abs() + 1)).astype("float32")
    return data


t0 = time.time()
data = construire_variables(ech)
print(f"Variables construites en {time.time() - t0:.0f} s | doublons supprimés : {len(ech) - len(data)}")

### A.1 Contrôle anti-fuite (nouvelles variables comprises)

In [ ]:
def controle_fuite(data, ech, n_cartes=3, seed=0):
    """Recalcule les variables d'une carte en retirant son futur : le passé ne doit pas bouger."""
    cols = ["nb_tx_1h", "nb_tx_24h", "nb_etranger_24h", "nb_nouveaux_marchands_24h", "montant_24h",
            "moy_30j", "nouveau_marchand", "nouvelle_ville", "h_depuis_prec"]
    cartes = data.loc[data["fraude"] == 1, "carte"].drop_duplicates().sample(n_cartes, random_state=seed)
    pire = 0.0
    for c in cartes:
        u, k = map(int, c.split("_"))
        brut = ech[(ech["User"] == u) & (ech["Card"] == k)].sort_values("date")
        coupe = brut["date"].iloc[len(brut) // 2]
        partiel = construire_variables(brut[brut["date"] <= coupe])
        complet = data[(data["carte"] == c) & (data["date"] <= coupe)]
        ecart = (complet[cols].reset_index(drop=True).astype(float)
                 - partiel[cols].reset_index(drop=True).astype(float)).abs().max().max()
        pire = max(pire, ecart)
    print(f"Écart maximal en retirant le futur ({n_cartes} cartes) : {pire:.6f}  -> doit être ~0")

controle_fuite(data, ech)

### A.2 Signal des nouvelles variables (entraînement seulement)

In [ ]:
tr = data[data["partie"] == "entraînement"]
def signal(col, bins=None):
    s = pd.cut(tr[col], bins) if bins is not None else tr[col]
    t = tr.groupby(s, observed=True)["fraude"].agg(transactions="size", taux="mean")
    t["taux"] = (t["taux"] * 100).round(3)
    return t.to_string()

print("Nouvelle ville :\n", signal("nouvelle_ville"))
print("\nTransactions à l'étranger sur 24 h :\n", signal("nb_etranger_24h", [-1, 0, 1, 3, np.inf]))
print("\nNouveaux marchands sur 24 h :\n", signal("nb_nouveaux_marchands_24h", [-1, 0, 1, 3, np.inf]))
print("\nMontant dépensé sur 24 h ($) :\n", signal("montant_24h", [-1, 0, 50, 200, 500, 1000, np.inf]))

In [ ]:
VARIABLES = [
    "heure_sin", "heure_cos", "jour_semaine", "week_end", "nuit",
    "montant", "log_montant", "remboursement", "en_ligne", "bande", "etranger",
    "err_cvv", "err_expiration", "err_numero", "err_pin", "err_solde", "err_technique", "err_code_postal", "une_erreur",
    "h_depuis_prec", "nb_tx_1h", "nb_tx_24h", "nb_tx_30j", "nb_err_24h", "moy_30j", "ratio_montant_30j",
    "nb_etranger_24h", "nb_nouveaux_marchands_24h", "montant_24h",
    "nouveau_marchand", "nouvelle_categorie", "nouvelle_ville", "part_en_ligne_passee",
]
CATEGORIELLE = "mcc_groupe"
# "Merchant Name" est gardé pour la comparaison « risque du marchand », jamais dans le modèle principal
GARDER = ["date", "carte", "partie", "fraude", "Merchant Name", CATEGORIELLE] + VARIABLES

final = data.loc[data["partie"] != "chauffe", GARDER].sort_values("date").reset_index(drop=True)
final.to_parquet(f"{DOSSIER}/variables_v2.parquet", index=False)
print(f"variables_v2.parquet : {len(final):,} lignes, {len(VARIABLES)} variables numériques + {CATEGORIELLE}")
print(final.groupby("partie")["fraude"].agg(transactions="size", fraudes="sum").to_string())

---
# Partie B – Module 3 : modélisation

## B.0 Préparation et règles d'évaluation

- **Budget d'analyse** : `BUDGET_JOUR` alertes par jour pour ce portefeuille de 500 clients, donc
  *k* = budget × nombre de jours de la période évaluée. On le reprendra au Module 4.
- **Encodage du MCC** : catégorie native pour XGBoost ; pour les autres modèles, **taux de fraude lissé du groupe
  MCC calculé sur l'entraînement seul** (`mcc_risque`).
- Valeurs manquantes (première transaction d'une carte) : médianes de l'entraînement pour les modèles qui ne les gèrent pas.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

BUDGET_JOUR = 10          # alertes par jour que l'équipe peut traiter (hypothèse à justifier dans le rapport)
GRAINE = 42

tr = final[final["partie"] == "entraînement"].reset_index(drop=True)
va = final[final["partie"] == "validation"].reset_index(drop=True)
y_tr, y_va = tr["fraude"].to_numpy(), va["fraude"].to_numpy()
jours_va = va["date"].dt.normalize().nunique()
K_VA = BUDGET_JOUR * jours_va
print(f"Validation : {jours_va} jours -> k = {K_VA:,} alertes ({K_VA / len(va) * 100:.2f} % des transactions)")
print(f"Précision@k maximale possible : {min(1, y_va.sum() / K_VA) * 100:.1f} % ({y_va.sum()} fraudes)")

# Encodage du MCC appris sur l'entraînement seul
taux_global = y_tr.mean()
stat = tr.groupby(CATEGORIELLE)["fraude"].agg(["sum", "count"])
LISSAGE = 100
mcc_risque = (stat["sum"] + LISSAGE * taux_global) / (stat["count"] + LISSAGE)
categories_mcc = sorted(tr[CATEGORIELLE].unique())

def matrice_xgb(df):
    X = df[VARIABLES].copy()
    X[CATEGORIELLE] = pd.Categorical(df[CATEGORIELLE], categories=categories_mcc)
    return X

medianes = tr[VARIABLES].median()
def matrice_num(df):
    X = df[VARIABLES].fillna(medianes).copy()
    X["mcc_risque"] = df[CATEGORIELLE].map(mcc_risque).fillna(taux_global).astype("float32")
    return X

Xtr_xgb, Xva_xgb = matrice_xgb(tr), matrice_xgb(va)
Xtr_num, Xva_num = matrice_num(tr), matrice_num(va)
print("Matrices prêtes :", Xtr_xgb.shape, Xtr_num.shape)

In [ ]:
def evaluer(nom, y, score, k=K_VA, temps=None):
    ordre = np.argsort(-score)[:k]
    vp = y[ordre].sum()
    return {"modèle": nom,
            "PR-AUC": average_precision_score(y, score),
            "ROC-AUC": roc_auc_score(y, score),
            f"précision@{k}": vp / k,
            f"rappel@{k}": vp / y.sum(),
            "temps (s)": temps}

def corriger_proba(ps, beta):
    """Correction après sous-échantillonnage (cours, diapo 156) : p = beta*ps / (beta*ps - ps + 1)."""
    return beta * ps / (beta * ps - ps + 1)

def sous_echantillonner(X, y, ratio, seed):
    """Garde toutes les fraudes et `ratio` normales par fraude (algorithme du cours). Renvoie aussi beta."""
    rng = np.random.default_rng(seed)
    fraudes, normales = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    m = min(len(normales), ratio * len(fraudes))
    garde = np.concatenate([fraudes, rng.choice(normales, m, replace=False)])
    rng.shuffle(garde)
    return X.iloc[garde], y[garde], m / len(normales)

resultats = []

## B.1 Stratégie contre le déséquilibre (XGBoost, comparaison en validation)

Ordre conseillé par le cours : rien → pondération → sous-échantillonnage (1:5, 1:10, 1:20, 3 graines) → SMOTE.
Le rééchantillonnage ne touche **que l'entraînement**. Critère : **PR-AUC en validation**.

In [ ]:
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

def xgb(**kw):
    base = dict(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
                tree_method="hist", enable_categorical=True, eval_metric="aucpr", n_jobs=-1, random_state=GRAINE)
    base.update(kw)
    return XGBClassifier(**base)

strategies = []
def essai(nom, X, y, Xva, **kw):
    t0 = time.time()
    m = xgb(**kw).fit(X, y)
    s = m.predict_proba(Xva)[:, 1]
    r = evaluer(nom, y_va, s, temps=round(time.time() - t0, 1))
    strategies.append(r); print(f"{nom:32s} PR-AUC {r['PR-AUC']:.4f}  ({r['temps (s)']} s)")
    return r

essai("aucune (seuil seulement)", Xtr_xgb, y_tr, Xva_xgb)
essai("pondération des classes", Xtr_xgb, y_tr, Xva_xgb, scale_pos_weight=(y_tr == 0).sum() / y_tr.sum())
for ratio in [5, 10, 20]:
    scores = []
    for seed in range(3):
        Xs, ys, beta = sous_echantillonner(Xtr_xgb, y_tr, ratio, seed)
        t0 = time.time(); m = xgb().fit(Xs, ys)
        scores.append(average_precision_score(y_va, m.predict_proba(Xva_xgb)[:, 1]))
    strategies.append({"modèle": f"sous-échantillonnage 1:{ratio}", "PR-AUC": np.mean(scores),
                       "écart-type": np.std(scores), "temps (s)": round(time.time() - t0, 1)})
    print(f"sous-échantillonnage 1:{ratio:<12} PR-AUC {np.mean(scores):.4f} ± {np.std(scores):.4f} (3 graines)")

# SMOTE : nécessite des variables numériques -> matrice numérique, après un sous-échantillonnage 1:20 pour rester rapide
Xs, ys, beta = sous_echantillonner(Xtr_num, y_tr, 20, 0)
Xs, ys = SMOTE(sampling_strategy=0.2, random_state=GRAINE).fit_resample(Xs, ys)
essai("1:20 puis SMOTE jusqu'à 1:5", Xs, ys, Xva_num, enable_categorical=False)

tab_strat = pd.DataFrame(strategies).set_index("modèle")
display(tab_strat[["PR-AUC", "écart-type", "temps (s)"]].round(4))

In [ ]:
# Choix de la stratégie : meilleure PR-AUC. Comme le conseille le cours, SMOTE n'est retenu que s'il apporte
# un gain mesurable (> 0,005 de PR-AUC) par rapport à la meilleure stratégie sans données artificielles.
sans_smote = tab_strat.drop(index=[i for i in tab_strat.index if "SMOTE" in i])
meilleure = sans_smote["PR-AUC"].idxmax()
smote = [i for i in tab_strat.index if "SMOTE" in i][0]
if tab_strat.loc[smote, "PR-AUC"] > sans_smote["PR-AUC"].max() + 0.005:
    meilleure = smote
print("Stratégie retenue :", meilleure)

STRATEGIE = {"nom": meilleure}
if meilleure.startswith("sous-échantillonnage"):
    STRATEGIE["ratio"] = int(meilleure.split(":")[1])
UTILISE_NUM = "SMOTE" in meilleure          # SMOTE impose la matrice numérique (MCC encodé)
Xtr_base, Xva_base = (Xtr_num, Xva_num) if UTILISE_NUM else (Xtr_xgb, Xva_xgb)

def donnees_strategie(Xbase, seed=0):
    """Applique la stratégie retenue à l'entraînement. Renvoie X, y, beta, paramètres XGBoost en plus.
    beta corrige les probabilités : cote réelle = beta x cote apprise (cours, diapo 156)."""
    n_f = y_tr.sum()
    if "SMOTE" in STRATEGIE["nom"]:
        Xs, ys, beta = sous_echantillonner(Xbase, y_tr, 20, seed)
        Xs, ys = SMOTE(sampling_strategy=0.2, random_state=GRAINE).fit_resample(Xs, ys)
        return Xs, ys, beta * n_f / ys.sum(), {}            # les fraudes ont été multipliées par ys.sum()/n_f
    if "ratio" in STRATEGIE:
        Xs, ys, beta = sous_echantillonner(Xbase, y_tr, STRATEGIE["ratio"], seed)
        return Xs, ys, beta, {}
    if STRATEGIE["nom"].startswith("pondération"):
        spw = (y_tr == 0).sum() / n_f
        return Xbase, y_tr, 1 / spw, {"scale_pos_weight": spw}
    return Xbase, y_tr, 1.0, {}

## B.2 XGBoost : réglage des hyperparamètres (validation)

Petite recherche aléatoire, avec la stratégie retenue. Critère : PR-AUC en validation.

In [ ]:
rng = np.random.default_rng(GRAINE)
grille = [dict(max_depth=int(rng.choice([4, 6, 8])), learning_rate=float(rng.choice([0.03, 0.06, 0.1])),
               n_estimators=int(rng.choice([300, 600])), min_child_weight=float(rng.choice([1, 5, 20])),
               subsample=0.8, colsample_bytree=float(rng.choice([0.6, 0.8])))
          for _ in range(6)]

Xs, ys, BETA, extra = donnees_strategie(Xtr_base)
extra["enable_categorical"] = not UTILISE_NUM
essais = []
for i, p in enumerate(grille):
    t0 = time.time()
    m = xgb(**{**p, **extra}).fit(Xs, ys)
    pr = average_precision_score(y_va, m.predict_proba(Xva_base)[:, 1])
    essais.append({**p, "PR-AUC": pr, "temps (s)": round(time.time() - t0, 1)})
    print(f"essai {i + 1}/6 : PR-AUC {pr:.4f}  {p}")
essais = pd.DataFrame(essais).sort_values("PR-AUC", ascending=False)
PARAMS_XGB = essais.iloc[0].drop(["PR-AUC", "temps (s)"]).to_dict()
PARAMS_XGB = {k: (int(v) if k in ("max_depth", "n_estimators") else float(v)) for k, v in PARAMS_XGB.items()}
print("\nMeilleurs paramètres :", PARAMS_XGB)

In [ ]:
t0 = time.time()
modele_xgb = xgb(**{**PARAMS_XGB, **extra}).fit(Xs, ys)
s_xgb = modele_xgb.predict_proba(Xva_base)[:, 1]
if BETA != 1:
    s_xgb = corriger_proba(s_xgb, BETA)     # probabilités remises à l'échelle réelle (utile au Module 4)
print(f"Stratégie : {STRATEGIE['nom']} | beta = {BETA:.5f} | probabilité moyenne prédite {s_xgb.mean():.5f} "
      f"vs taux réel {y_va.mean():.5f}")
resultats.append(evaluer("XGBoost", y_va, s_xgb, temps=round(time.time() - t0, 1)))
print(resultats[-1])

## B.3 Régression logistique pondérée (référence supervisée simple)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

t0 = time.time()
modele_lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
modele_lr.fit(Xtr_num, y_tr)
s_lr = modele_lr.predict_proba(Xva_num)[:, 1]
resultats.append(evaluer("Régression logistique pondérée", y_va, s_lr, temps=round(time.time() - t0, 1)))
print(resultats[-1])

## B.4 Isolation Forest (non supervisé, détection d'anomalies)

Entraîné sur **toutes** les transactions d'entraînement, **sans** la colonne `fraude`.
Les étiquettes ne servent qu'à l'évaluation.

In [ ]:
from sklearn.ensemble import IsolationForest

t0 = time.time()
modele_if = IsolationForest(n_estimators=200, max_samples=256, random_state=GRAINE, n_jobs=-1).fit(Xtr_num)
s_if = -modele_if.score_samples(Xva_num)           # plus grand = plus suspect
resultats.append(evaluer("Isolation Forest", y_va, s_if, temps=round(time.time() - t0, 1)))
print(resultats[-1])

## B.5 Autoencodeur (non supervisé, détection de nouveauté)

Entraîné uniquement sur des transactions **normales** de l'entraînement (on utilise les étiquettes une seule fois,
pour filtrer). Score = erreur de reconstruction.

In [ ]:
from sklearn.neural_network import MLPRegressor

normales = Xtr_num[y_tr == 0].sample(200_000, random_state=GRAINE) if (y_tr == 0).sum() > 200_000 else Xtr_num[y_tr == 0]
sc_ae = StandardScaler().fit(normales)
t0 = time.time()
modele_ae = MLPRegressor(hidden_layer_sizes=(24, 8, 24), max_iter=60, early_stopping=True, random_state=GRAINE)
modele_ae.fit(sc_ae.transform(normales), sc_ae.transform(normales))
Z = sc_ae.transform(Xva_num)
s_ae = ((np.clip(Z, -10, 10) - modele_ae.predict(Z)) ** 2).mean(axis=1)
resultats.append(evaluer("Autoencodeur", y_va, s_ae, temps=round(time.time() - t0, 1)))
print(resultats[-1])

## B.6 LOF (optionnel, plus lent)

`novelty=True` pour noter des transactions nouvelles. Entraîné sur un échantillon de normales (LOF est lent sur de gros volumes).
Passe `FAIRE_LOF = True` pour l'exécuter (compter environ 5 à 15 minutes).

In [ ]:
FAIRE_LOF = False
if FAIRE_LOF:
    from sklearn.neighbors import LocalOutlierFactor
    norm_lof = Xtr_num[y_tr == 0].sample(30_000, random_state=GRAINE)
    sc_lof = StandardScaler().fit(norm_lof)
    t0 = time.time()
    modele_lof = LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1).fit(sc_lof.transform(norm_lof))
    s_lof = -modele_lof.score_samples(sc_lof.transform(Xva_num))
    resultats.append(evaluer("LOF", y_va, s_lof, temps=round(time.time() - t0, 1)))
    print(resultats[-1])

## B.7 Tableau comparatif (validation 2018)

In [ ]:
tableau = pd.DataFrame(resultats).set_index("modèle").sort_values("PR-AUC", ascending=False)
print(f"Référence au hasard : PR-AUC = taux de fraude = {y_va.mean():.4f}")
display(tableau.round(4))

from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(7, 4.5))
scores_va = {"XGBoost": s_xgb, "Régression logistique": s_lr, "Isolation Forest": s_if, "Autoencodeur": s_ae}
if FAIRE_LOF: scores_va["LOF"] = s_lof
for nom, s in scores_va.items():
    p, r, _ = precision_recall_curve(y_va, s)
    ax.plot(r, p, label=f"{nom} (PR-AUC {average_precision_score(y_va, s):.3f})")
ax.axhline(y_va.mean(), color="grey", ls=":", label="hasard")
ax.set_xlabel("rappel"); ax.set_ylabel("précision"); ax.set_title("Courbes précision-rappel – validation")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## B.8 Comparaison : avec « risque du marchand » (raccourci possible)

Pour mesurer le raccourci repéré au Module 1 : un taux de fraude par marchand, calculé **uniquement avec les étiquettes
de l'entraînement**, ajouté à XGBoost. Ce modèle n'est **pas** retenu : il sert à discuter.

In [ ]:
stat_m = tr.groupby("Merchant Name")["fraude"].agg(["sum", "count"])
risque_m = (stat_m["sum"] + 50 * taux_global) / (stat_m["count"] + 50)
Xtr_m, Xva_m = Xtr_base.copy(), Xva_base.copy()
Xtr_m["risque_marchand"] = tr["Merchant Name"].map(risque_m).astype("float32")   # NB : vu de l'entraînement lui-même
Xva_m["risque_marchand"] = va["Merchant Name"].map(risque_m).fillna(taux_global).astype("float32")

Xs_m, ys_m, _, _ = donnees_strategie(Xtr_m)        # même stratégie, même graine que le modèle principal
m_m = xgb(**{**PARAMS_XGB, **extra}).fit(Xs_m, ys_m)
s_m = m_m.predict_proba(Xva_m)[:, 1]
comp = pd.DataFrame([evaluer("XGBoost (modèle principal)", y_va, s_xgb),
                     evaluer("XGBoost + risque du marchand", y_va, s_m)]).set_index("modèle")
display(comp.round(4))
imp = pd.Series(m_m.feature_importances_, index=Xtr_m.columns).sort_values(ascending=False)
print(f"Part d'importance (gain) de risque_marchand : {imp['risque_marchand'] / imp.sum() * 100:.1f} % "
      f"(rang {list(imp.index).index('risque_marchand') + 1})")

## B.9 Sauvegarde pour le Module 4

In [ ]:
joblib.dump({"xgb": modele_xgb, "lr": modele_lr, "iforest": modele_if, "ae": modele_ae, "sc_ae": sc_ae},
            f"{DOSSIER}/modeles_v1.joblib")
config = {"VARIABLES": VARIABLES, "CATEGORIELLE": CATEGORIELLE, "categories_mcc": categories_mcc,
          "mcc_risque": mcc_risque.to_dict(), "taux_global": float(taux_global), "medianes": medianes.to_dict(),
          "BUDGET_JOUR": BUDGET_JOUR, "STRATEGIE": STRATEGIE, "UTILISE_NUM": UTILISE_NUM, "BETA": float(BETA), "PARAMS_XGB": PARAMS_XGB}
json.dump(config, open(f"{DOSSIER}/config_v1.json", "w"), ensure_ascii=False, indent=1, default=str)
pd.DataFrame({"date": va["date"], "fraude": y_va, "montant": va["montant"], **{k: v for k, v in scores_va.items()}}) \
  .to_parquet(f"{DOSSIER}/scores_validation_v1.parquet", index=False)
tableau.to_csv(f"{DOSSIER}/tableau_comparatif_v1.csv")
print("Sauvegardé : modeles_v1.joblib, config_v1.json, scores_validation_v1.parquet, tableau_comparatif_v1.csv")

## À rapporter pour l'analyse

Sorties texte de : **A.1, A.2, B.0, B.1 (tableau des stratégies), B.2, B.7 (tableau comparatif), B.8**,
et une capture des courbes précision-rappel.